In [1]:
import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import datetime, timedelta

fake = Faker()

# Make results reproducible
Faker.seed(42)
random.seed(42)
np.random.seed(42)

In [2]:
n_customers = 500  # you can bump this later

industries = ["FinTech", "HealthTech", "E-commerce", "SaaS", "EdTech", "Logistics"]
company_sizes = ["SMB", "Mid-Market", "Enterprise"]
regions = ["North America", "Europe", "Asia"]
channels = ["Organic", "Paid Ads", "Referral", "Partner", "Outbound"]

customers = []

for i in range(1, n_customers + 1):
    signup_date = fake.date_between(start_date="-3y", end_date="today")
    customers.append({
        "customer_id": i,
        "company_name": fake.company(),
        "industry": random.choice(industries),
        "company_size": random.choice(company_sizes),
        "region": random.choice(regions),
        "acquisition_channel": random.choice(channels),
        "signup_date": signup_date
    })

customers_df = pd.DataFrame(customers)
customers_df.head()

,customer_id,company_name,industry,company_size,region,acquisition_channel,signup_date
0,1,Johnson LLC,Logistics,SMB,North America,Referral,2026-04-06
1,2,Taylor Inc,HealthTech,SMB,North America,Organic,2024-08-01
2,3,"Mcclain, Miller and Henderson",Logistics,Enterprise,Asia,Organic,2026-06-03
3,4,Robinson PLC,EdTech,Mid-Market,North America,Organic,2023-09-02
4,5,Hoffman Ltd,FinTech,SMB,North America,Outbound,2026-02-07


In [3]:
plans = ["Starter", "Growth", "Pro", "Enterprise"]

plan_prices = {
    "Starter": 49,
    "Growth": 99,
    "Pro": 249,
    "Enterprise": 599
}

subscriptions = []

for _, row in customers_df.iterrows():
    plan = random.choices(
        plans,
        weights=[0.35, 0.30, 0.20, 0.15],  # more Starter/Growth, fewer Enterprise
        k=1
    )[0]
    
    billing_cycle = random.choice(["Monthly", "Annual"])
    
    seats = {
        "Starter": random.randint(1, 5),
        "Growth": random.randint(5, 20),
        "Pro": random.randint(20, 80),
        "Enterprise": random.randint(80, 300)
    }[plan]
    
    monthly_price = plan_prices[plan]
    
    start_date = pd.to_datetime(row["signup_date"])
    renewal_date = start_date + pd.Timedelta(
        days=30 if billing_cycle == "Monthly" else 365
    )
    
    subscriptions.append({
        "subscription_id": f"S{row['customer_id']:04d}",
        "customer_id": row["customer_id"],
        "plan_name": plan,
        "billing_cycle": billing_cycle,
        "monthly_price": monthly_price,
        "seats": seats,
        "start_date": start_date.date(),
        "renewal_date": renewal_date.date(),
        "status": "Active"
    })

subscriptions_df = pd.DataFrame(subscriptions)
subscriptions_df.head()

,subscription_id,customer_id,plan_name,billing_cycle,monthly_price,seats,start_date,renewal_date,status
0,S0001,1,Growth,Annual,99,14,2026-04-06,2027-04-06,Active
1,S0002,2,Starter,Annual,49,1,2024-08-01,2025-08-01,Active
2,S0003,3,Growth,Monthly,99,12,2026-06-03,2026-07-03,Active
3,S0004,4,Starter,Monthly,49,5,2023-09-02,2023-10-02,Active
4,S0005,5,Growth,Monthly,99,16,2026-02-07,2026-03-09,Active


In [4]:
customers_df.to_csv("../data/customers.csv", index=False)
subscriptions_df.to_csv("../data/subscriptions.csv", index=False)

print("Saved customers.csv and subscriptions.csv")
print("customers_df shape:", customers_df.shape)
print("subscriptions_df shape:", subscriptions_df.shape)

Saved customers.csv and subscriptions.csv
customers_df shape: (500, 7)
subscriptions_df shape: (500, 9)


In [5]:
# Generate monthly usage data

# Choose date range for usage history (e.g., last 24 months)
usage_months = pd.date_range(
    end=pd.to_datetime("today").normalize(),
    periods=24,
    freq="MS"  # Month start
)

usage_records = []

for _, row in customers_df.iterrows():
    # Use signup date so earlier customers have more months
    start = pd.to_datetime(row["signup_date"]).to_period("M").to_timestamp()
    
    for month in usage_months:
        if month < start:
            continue  # no usage before signup
        
        # Simple engagement score by plan & segment
        base_sessions = {
            "Starter": 20,
            "Growth": 50,
            "Pro": 120,
            "Enterprise": 300
        }
        
        # Find this customer's plan
        sub = subscriptions_df.loc[
            subscriptions_df["customer_id"] == row["customer_id"]
        ].iloc[0]
        
        plan_name = sub["plan_name"]
        
        # Add some randomness + noise by company size
        sessions = base_sessions[plan_name] + np.random.randint(-10, 30)
        sessions = max(sessions, 0)
        
        active_users = max(
            1,
            int(sub["seats"] * np.random.uniform(0.3, 0.9))
        )
        
        feature_adoption_score = np.clip(
            np.random.normal(loc=0.6, scale=0.15),
            0.0,
            1.0
        )
        
        api_calls = int(
            sessions * np.random.uniform(2, 8)
        )
        
        usage_records.append({
            "customer_id": row["customer_id"],
            "month": month.date(),
            "active_users": active_users,
            "sessions": sessions,
            "feature_adoption_score": round(feature_adoption_score, 2),
            "api_calls": api_calls
        })

usage_df = pd.DataFrame(usage_records)
usage_df.head()

,customer_id,month,active_users,sessions,feature_adoption_score,api_calls
0,1,2026-04-01,10,78,0.68,435
1,1,2026-05-01,5,58,0.51,275
2,1,2026-06-01,5,75,0.53,474
3,1,2026-07-01,5,69,0.62,213
4,2,2024-08-01,1,30,0.60,131


In [6]:
usage_df.to_csv("../data/usage_monthly.csv", index=False)

print("Saved usage_monthly.csv")
print("usage_df shape:", usage_df.shape)

Saved usage_monthly.csv
usage_df shape: (8159, 6)


In [7]:
customers_df.shape, subscriptions_df.shape, usage_df.shape

((500, 7), (500, 9), (8159, 6))

In [8]:
# Generate invoices (billing / revenue data)

invoice_records = []

for _, sub in subscriptions_df.iterrows():
    customer_id = sub["customer_id"]
    start_date = pd.to_datetime(sub["start_date"])
    billing_cycle = sub["billing_cycle"]
    monthly_price = sub["monthly_price"]
    seats = sub["seats"]
    
    # base per-seat monthly revenue
    base_mrr = monthly_price * seats
    
    # generate invoices for ~24 months or until today
    if billing_cycle == "Monthly":
        invoice_dates = pd.date_range(
            start=start_date,
            end=pd.to_datetime("today").normalize(),
            freq="MS"
        )
        multiplier = 1
    else:
        invoice_dates = pd.date_range(
            start=start_date,
            end=pd.to_datetime("today").normalize(),
            freq="12MS"
        )
        multiplier = 12
    
    for inv_date in invoice_dates:
        billed_amount = base_mrr * multiplier
        
        # random discounts and payment issues
        discount_amount = 0
        if np.random.rand() < 0.15:  # 15% of invoices get discounts
            discount_amount = billed_amount * np.random.uniform(0.05, 0.25)
        
        # simulate payment issues
        if np.random.rand() < 0.08:
            payment_status = "Failed"
            paid_amount = 0
        else:
            payment_status = "Paid"
            paid_amount = billed_amount - discount_amount
        
        invoice_records.append({
            "invoice_id": f"INV-{customer_id:04d}-{inv_date.strftime('%Y%m')}",
            "customer_id": customer_id,
            "invoice_month": inv_date.date(),
            "billed_amount": round(billed_amount, 2),
            "discount_amount": round(discount_amount, 2),
            "paid_amount": round(paid_amount, 2),
            "payment_status": payment_status
        })

invoices_df = pd.DataFrame(invoice_records)
invoices_df.head()

,invoice_id,customer_id,invoice_month,billed_amount,discount_amount,paid_amount,payment_status
0,INV-0001-202605,1,2026-05-01,16632,1267.98,15364.02,Paid
1,INV-0002-202408,2,2024-08-01,588,0.00,588.00,Paid
2,INV-0002-202508,2,2025-08-01,588,43.26,0.00,Failed
3,INV-0003-202607,3,2026-07-01,1188,0.00,1188.00,Paid
4,INV-0004-202310,4,2023-10-01,245,0.00,245.00,Paid


In [9]:
invoices_df.to_csv("../data/invoices.csv", index=False)

print("Saved invoices.csv")
print("invoices_df shape:", invoices_df.shape)

Saved invoices.csv
invoices_df shape: (4945, 7)
